# Lab 1.2 — Server Health CLI Tool

## 0 — Setup

Run this cell first. It installs `requests` and creates the `servers.json` input file.

In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'requests', '-q'])

import json, pathlib

servers_data = [
    {"name": "api-prod-1",       "host": "httpbin.org", "port": 443, "protocol": "https", "health_path": "/status/200"},
    {"name": "api-prod-2",       "host": "httpbin.org", "port": 443, "protocol": "https", "health_path": "/status/503"},
    {"name": "slow-server",      "host": "httpbin.org", "port": 443, "protocol": "https", "health_path": "/delay/6"},
    {"name": "unreachable",      "host": "10.255.255.1", "port": 8080, "protocol": "http", "health_path": "/health"}
]

pathlib.Path('servers.json').write_text(json.dumps(servers_data, indent=2))
print('✅ servers.json created with', len(servers_data), 'entries')

✅ servers.json created with 4 entries


---
### Task 1 — Load the config file

###  Concept: `json` and `pathlib`

`pathlib.Path` is the modern way to work with files in Python. It works on all operating systems.

```python
import json, pathlib

# Read a file as text, then parse JSON
raw = pathlib.Path('config.json').read_text()
data = json.loads(raw)

# Write JSON to a file
pathlib.Path('output.json').write_text(json.dumps(data, indent=2))
```

Always handle errors specifically:
- `FileNotFoundError` — the file doesn't exist
- `json.JSONDecodeError` — the file exists but isn't valid JSON

In [4]:
import json
import pathlib
import sys

# ✏️ YOUR TURN
# Write a function load_servers(path: str) -> list[dict] that:
#   1. Reads the JSON file at the given path
#   2. Returns the parsed list of server dicts
#   3. Prints a clear error and exits if the file is not found or invalid JSON

def load_servers(path: str) -> list[dict]:
    """Load and return the list of servers from a JSON file."""
    try:
        with open(path, 'r') as file:
            return json.load(file)
    except FileNotFoundError:
        print(f"Error: File not found: {path}")
        sys.exit(1)
    except json.JSONDecodeError:
        print(f"Error: Invalid JSON in file: {path}")
        sys.exit(1)

# Test your function
servers = load_servers('servers.json')
print(f'Loaded {len(servers)} servers')
print('First server:', servers[0])

Loaded 4 servers
First server: {'name': 'api-prod-1', 'host': 'httpbin.org', 'port': 443, 'protocol': 'https', 'health_path': '/status/200'}


---
## Task 2 — Check a single server

### 📖 Concept: `requests` + error handling

The `requests` library makes HTTP calls simple. Always set a `timeout` — otherwise your script hangs forever if a server doesn't respond.

```python
import requests, time

start = time.time()
try:
    resp = requests.get('https://example.com/health', timeout=5)
    elapsed_ms = (time.time() - start) * 1000
    print(resp.status_code, elapsed_ms)
except requests.exceptions.ConnectionError:
    print('Could not connect')
except requests.exceptions.Timeout:
    print('Request timed out')
```

**Status rules for our tool:**
- `UP` — HTTP 200 and response time ≤ 500 ms
- `DEGRADED` — HTTP 200 but slow, OR a non-200 HTTP response
- `DOWN` — connection error or timeout

In [15]:
import requests
import time

# ✏️ YOUR TURN
# Write check_server(server: dict) -> dict
# It should return a dict with keys:
#   name, url, status, response_time_ms, status_code
# For DOWN servers: response_time_ms=None, status_code=None, add an 'error' key

def check_server(server: dict) -> dict:
    """Check the health of a single server and return a result dict."""
    url = f"{server['protocol']}://{server['host']}:{server['port']}{server['health_path']}"
    start = time.time()
    
    try:
        response = requests.get(url, timeout=5)
        response_time_ms = (time.time() - start) * 1000
        if response_time_ms <= 500:
            status = "UP"
        else:
            status = "DEGRADED"
        return {
            "name": server["name"],
            "url": url,
            "status": status,
            "response_time_ms": response_time_ms,
            "status_code": response.status_code,
        }

    except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as e:
        return {
            "name": server["name"],
            "url": url,
            "status": "DOWN",
            "response_time_ms": None,
            "status_code": None,
            "error": str(e)
        }

# Quick test with the first (healthy) server
result = check_server(servers[0])
print(result)

{'name': 'api-prod-1', 'url': 'https://httpbin.org:443/status/200', 'status': 'DEGRADED', 'response_time_ms': 672.980546951294, 'status_code': 503}


---
## Task 3 — Build the summary report

### 📖 Concept: `datetime` for timestamps

```python
from datetime import datetime
now = datetime.now().isoformat(timespec='seconds')
# '2026-05-30T09:45:00'
```

In [38]:
from datetime import datetime

# ✏️ YOUR TURN
# Write run_health_check(servers: list[dict]) -> dict
# It should:
#   1. Call check_server() for each server
#   2. Print a one-line status per server as it runs
#   3. Return a report dict with: checked_at, total, up, degraded, down, results

def run_health_check(servers: list[dict]) -> dict:
    results = []
    for server in servers:
        result = check_server(server)
        results.append(result)
        print(f"{result['name']}: {result['status']} (Response Time: {result['response_time_ms']} ms, Status Code: {result['status_code']})")

    report = {
        "checked_at": datetime.now().isoformat(),
        "total": len(servers),
        "up": sum(1 for r in results if r['status'] == 'UP'),
        "degraded": sum(1 for r in results if r['status'] == 'DEGRADED'),
        "down": sum(1 for r in results if r['status'] == 'DOWN'),
        "results": results
    }
    return report


report = run_health_check(servers)
print(f"\n{'='*40}")
print(f"  Report - {report['checked_at'][:19]}")
print(f"{'='*40}")
print(f"  Total:    {report['total']}")
print(f"  UP:       {report['up']}")
print(f"  DEGRADED: {report['degraded']}")
print(f"  DOWN:     {report['down']}")
print(f"{'='*40}")

for result in report['results']:
    print("result :", result)



api-prod-1: DEGRADED (Response Time: 818.3300495147705 ms, Status Code: 503)
api-prod-2: DEGRADED (Response Time: 679.7025203704834 ms, Status Code: 503)
slow-server: DEGRADED (Response Time: 706.1734199523926 ms, Status Code: 503)
unreachable: DOWN (Response Time: None ms, Status Code: None)

  Report - 2026-06-29T14:02:21
  Total:    4
  UP:       0
  DEGRADED: 3
  DOWN:     1
result : {'name': 'api-prod-1', 'url': 'https://httpbin.org:443/status/200', 'status': 'DEGRADED', 'response_time_ms': 818.3300495147705, 'status_code': 503}
result : {'name': 'api-prod-2', 'url': 'https://httpbin.org:443/status/503', 'status': 'DEGRADED', 'response_time_ms': 679.7025203704834, 'status_code': 503}
result : {'name': 'slow-server', 'url': 'https://httpbin.org:443/delay/6', 'status': 'DEGRADED', 'response_time_ms': 706.1734199523926, 'status_code': 503}
result : {'name': 'unreachable', 'url': 'http://10.255.255.1:8080/health', 'status': 'DOWN', 'response_time_ms': None, 'status_code': None, 'error